# Session 2: energy from nutrients and the balance equation
**A. Real-world problem.** A food label lists grams of protein, carbohydrate and fat, and
also a number of kilocalories. Do they agree? And how many kilocalories does a person need
per day? This session builds the equations: label energy from grams, a resting-energy
estimate (Mifflin-St Jeor), activity multipliers, and a daily balance as a difference
equation. (General education only, not medical or dietary advice.)

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

KCAL_PER_G = {"protein": 4.0, "carbohydrate": 4.0, "fat": 9.0}   # Atwater general factors
KCAL_PER_KG = 7700.0                                              # rough rule, see Step 4

## Step 1: energy from grams
energy (kcal) = 4 x protein (g) + 4 x carbohydrate (g) + 9 x fat (g).
Printed labels use more detailed factors and rounding, so a small mismatch is normal.

In [ ]:
def label_kcal(protein_g, carb_g, fat_g):
    """Energy computed from grams (kcal)."""
    return (KCAL_PER_G["protein"] * protein_g + KCAL_PER_G["carbohydrate"] * carb_g
            + KCAL_PER_G["fat"] * fat_g)


def label_check(printed_kcal, protein_g, carb_g, fat_g):
    """Returns (computed kcal, printed - computed, difference as a percentage of printed)."""
    calc = label_kcal(protein_g, carb_g, fat_g)
    diff = printed_kcal - calc
    return calc, diff, 100.0 * diff / printed_kcal

## Step 2: resting energy (Mifflin-St Jeor) and activity multipliers
BMR (basal metabolic rate) is the energy the body uses at complete rest. The Mifflin-St Jeor
equation is a *population estimate* fitted to many people. For one person the true value can
be quite different, so treat the result as a starting guess, never a measurement.

    BMR = 10 m + 6.25 h - 5 a + s      (kcal/day)
    m = mass in kg, h = height in cm, a = age in years, s = +5 (male) or -161 (female)

Total daily energy use (TDEE) is BMR times an activity factor (PAL). These factors are common
textbook values, rounded.

In [ ]:
PAL = {"sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725}


def mifflin(sex, mass_kg, height_cm, age_years):
    if sex not in ("male", "female"):
        raise ValueError("sex must be 'male' or 'female'")
    s = 5.0 if sex == "male" else -161.0
    return 10.0 * mass_kg + 6.25 * height_cm - 5.0 * age_years + s


def tdee(bmr, pal):
    return bmr * pal

## Step 3: the balance as a difference equation
Let M[n] be body mass on day n (kg), I the energy eaten per day and E the energy used per day
(both kcal/day). A rough short-term rule says about 7700 kcal of surplus adds about 1 kg:

    M[n+1] = M[n] + (I - E) / 7700

Here E is held constant. Step 4 and Session 3 show why that is only a short-term approximation.

In [ ]:
def mass_series(m0, intake, expenditure, days):
    """Mass (kg) on days 0..days with constant intake and expenditure (kcal/day)."""
    m = np.empty(days + 1)
    m[0] = m0
    for n in range(days):
        m[n + 1] = m[n] + (intake - expenditure) / KCAL_PER_KG
    return m


def mass_change_rule(intake, expenditure, days):
    """Closed form of the same rule: (I - E) * days / 7700 (kg)."""
    return (intake - expenditure) * days / KCAL_PER_KG

## Step 4: limits (important)
The body adapts: when mass changes, expenditure changes, hormones and activity change too.
So 7700 kcal per kg is only a rough short-term approximation, and it gets worse the longer
you extend it. Session 3 shows this with a dynamic model.

## Step 5: a design problem for a made-up person
Ravi (made up): male, 19 years, 175 cm, 70 kg, moderate activity. Choose servings from a menu
so that his day's food energy matches his estimated energy use within 50 kcal.

In [ ]:
# (protein g, carbohydrate g, fat g) per serving, rounded and approximate
MENU = {
    "rice (1 cup)": (4, 45, 1),
    "dal (1 bowl)": (12, 30, 4),
    "chapati (1)": (3, 18, 3),
    "paneer (50 g)": (9, 2, 10),
    "milk (1 glass)": (8, 12, 8),
    "banana (1)": (1, 27, 0),
    "peanuts (30 g)": (8, 5, 14),
    "egg (1)": (6, 1, 5),
}
RAVI = dict(sex="male", mass_kg=70.0, height_cm=175.0, age=19, pal="moderate")


def menu_kcal(servings):
    """servings: dict name -> number of servings. Returns total kcal."""
    return float(sum(label_kcal(*MENU[k]) * n for k, n in servings.items()))


def plan_ok(servings, target_kcal, tol=50.0, max_per_item=4):
    """A plan is OK if the total is within tol of the target, uses at least 5 items, max 4 servings each."""
    used = [k for k, n in servings.items() if n > 0]
    return (abs(menu_kcal(servings) - target_kcal) <= tol and len(used) >= 5
            and all(0 <= n <= max_per_item for n in servings.values()))


RAVI_PLAN = {"rice (1 cup)": 2, "dal (1 bowl)": 3, "chapati (1)": 4, "paneer (50 g)": 2,
             "milk (1 glass)": 2, "banana (1)": 2, "peanuts (30 g)": 2, "egg (1)": 0}

## Step 6: see it
Left: mass over 30 days for three intakes with the constant-E rule (straight lines). Right:
label energy from grams against the printed value for the granola bar example.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    bmr = mifflin(RAVI["sex"], RAVI["mass_kg"], RAVI["height_cm"], RAVI["age"])
    E = tdee(bmr, PAL[RAVI["pal"]])
    print(f"Ravi: BMR {bmr:.2f} kcal/day, TDEE {E:.1f} kcal/day")
    print(f"Ravi's plan: {menu_kcal(RAVI_PLAN):.0f} kcal, ok = {plan_ok(RAVI_PLAN, E)}")
    print("granola bar:", [round(v, 2) for v in label_check(450, 5, 60, 20)])
    print(f"worked example: +259 kcal/day for 30 days -> {mass_change_rule(2900, E, 30):.2f} kg")

    days = 30
    fig, ax = plt.subplots(figsize=(6, 4))
    for intake in (E - 500, E, E + 500):
        ax.plot(np.arange(days + 1), mass_series(70.0, intake, E, days), label=f"intake {intake:.0f} kcal/day")
    ax.set_xlabel("day")
    ax.set_ylabel("body mass (kg)")
    ax.set_title("Constant-E rule: straight lines (short-term only)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "mass_rule.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(5, 4))
    parts = [4 * 5, 4 * 60, 9 * 20]
    ax2.bar(["protein", "carbohydrate", "fat"], parts, color=["#457b9d", "#e9c46a", "#e76f51"])
    ax2.set_ylabel("energy from this part (kcal)")
    ax2.set_title("Granola bar: 20 + 240 + 180 = 440 kcal (printed 450)")
    ax2.grid(alpha=0.3, axis="y")
    fig2.tight_layout()
    fig2.savefig(OUT / "label_parts.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Compute the energy from grams for a food label you have at home (or invent one) and compare it with the printed value.
2. **Stretch.** Change Ravi's activity factor to sedentary and rebuild a menu plan within 50 kcal of his new TDEE.
3. **Explain.** Explain, in two sentences, why a formula fitted to many people can be wrong for one person.

## Assessment
- **A. Conceptual.** Why is the 7700 kcal per kg rule only a short-term approximation?
- **B. Numerical.** An energy bar label says 8 g protein, 30 g carbohydrate and 12 g fat, and prints 270 kcal. Compute the energy from grams and the difference. Is the label wrong?
- **C. Graph and simulation interpretation.** In the 30-day graph the lines are straight. If a person's intake is 500 kcal above E, how much does the rule predict in 30 days and what is the slope?
- **D. Design and optimisation challenge.** Design: choose servings from the menu (rice 205, dal 204, chapati 111, paneer 134, milk 152, banana 112, peanuts 178, egg 73 kcal per serving) for Ravi so that the day's food is within 50 kcal of his TDEE, uses at least 5 different items and at most 4 servings of any item. State one trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("protein 4, carbohydrate 4, fat 9 kcal per gram", L.KCAL_PER_G == {"protein": 4.0, "carbohydrate": 4.0, "fat": 9.0}))
    calc, diff, pct = L.label_check(450, 5, 60, 20)
    checks.append(("hand calculation: 5 g protein, 60 g carbohydrate, 20 g fat = 20 + 240 + 180 = 440 kcal",
                   calc == 440.0 and L.label_kcal(5, 60, 20) == 440.0))
    checks.append(("printed 450 kcal is 10 kcal (2.2 percent) above the computed 440", diff == 10.0 and round(pct, 1) == 2.2))
    checks.append(("a label with only fat, 10 g, is 90 kcal", L.label_kcal(0, 0, 10) == 90.0))
    ravi = L.RAVI
    bmr = L.mifflin(ravi["sex"], ravi["mass_kg"], ravi["height_cm"], ravi["age"])
    checks.append(("Mifflin-St Jeor for Ravi (male, 70 kg, 175 cm, 19 y) is 1703.75 kcal/day", abs(bmr - 1703.75) < 1e-9))
    checks.append(("Mifflin-St Jeor for a made-up woman (60 kg, 165 cm, 20 y) is 1370.25 kcal/day",
                   abs(L.mifflin("female", 60, 165, 20) - 1370.25) < 1e-9))
    checks.append(("same body, male minus female is 166 kcal/day", abs(L.mifflin("male", 60, 165, 20) - L.mifflin("female", 60, 165, 20) - 166.0) < 1e-9))
    E = L.tdee(bmr, L.PAL[ravi["pal"]])
    checks.append(("Ravi's TDEE with factor 1.55 is 2640.8 kcal/day", round(E, 1) == 2640.8))
    checks.append(("sedentary factor 1.2 gives 2044.5 kcal/day for Ravi", abs(L.tdee(bmr, L.PAL["sedentary"]) - 2044.5) < 1e-9))
    checks.append(("worked example: eating 2900 kcal/day for 30 days adds about 1.01 kg by the 7700 rule",
                   round(L.mass_change_rule(2900, E, 30), 2) == 1.01))
    series = L.mass_series(70.0, 2900.0, E, 30)
    checks.append(("difference equation agrees with the closed form", abs(series[-1] - (70.0 + L.mass_change_rule(2900, E, 30))) < 1e-9))
    checks.append(("a surplus of exactly 7700 kcal adds 1 kg", abs(L.mass_change_rule(7700.0, 0.0, 1) - 1.0) < 1e-12))
    checks.append(("a deficit of 500 kcal/day for 28 days is -1.82 kg by the rule", round(L.mass_change_rule(0.0, 500.0, 28), 2) == -1.82))
    checks.append(("zero days gives the starting mass (edge case)", L.mass_series(70.0, 3000.0, 2000.0, 0).tolist() == [70.0]))
    raised = False
    try:
        L.mifflin("other", 60, 165, 20)
    except ValueError:
        raised = True
    checks.append(("an unknown sex value is rejected (edge case)", raised))
    checks.append(("Ravi's example menu plan has 2618 kcal, 22.8 kcal below his TDEE, and meets the rules",
                   L.menu_kcal(L.RAVI_PLAN) == 2618.0 and round(E - 2618.0, 1) == 22.8 and L.plan_ok(L.RAVI_PLAN, E)))
    checks.append(("a plan using only 3 items fails the rules even if the energy matches",
                   not L.plan_ok({"rice (1 cup)": 4, "dal (1 bowl)": 4, "chapati (1)": 4}, L.menu_kcal({"rice (1 cup)": 4, "dal (1 bowl)": 4, "chapati (1)": 4}))))
    checks.append(("surplus for Ravi at 2900 kcal/day is 259.2 kcal/day", round(2900.0 - E, 1) == 259.2))
    checks.append(("Mifflin terms: 6.25 x 175 = 1093.75 and 10 x 70 = 700", 6.25 * 175 == 1093.75 and 10 * 70 == 700))
    checks.append(("activity factors 1.2, 1.375, 1.55, 1.725", [L.PAL[k] for k in ("sedentary", "light", "moderate", "active")] == [1.2, 1.375, 1.55, 1.725]))
    calc2, diff2, pct2 = L.label_check(270, 8, 30, 12)
    checks.append(("exercise label: 8 g protein, 30 g carbohydrate, 12 g fat = 260 kcal against printed 270 (3.7 percent)",
                   calc2 == 260.0 and diff2 == 10.0 and round(pct2, 1) == 3.7))
    checks.append(("a surplus of 500 kcal/day for 30 days is +1.95 kg by the rule", round(L.mass_change_rule(E + 500.0, E, 30), 2) == 1.95))
    checks.append(("energy of the menu items: rice 205, dal 204, chapati 111, paneer 134, milk 152, banana 112, peanuts 178, egg 73 kcal",
                   [L.label_kcal(*L.MENU[k]) for k in ("rice (1 cup)", "dal (1 bowl)", "chapati (1)", "paneer (50 g)", "milk (1 glass)", "banana (1)", "peanuts (30 g)", "egg (1)")] == [205.0, 204.0, 111.0, 134.0, 152.0, 112.0, 178.0, 73.0]))
    checks.append(("Ravi's plan uses 7 different items", sum(1 for n in L.RAVI_PLAN.values() if n > 0) == 7))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
